# 060 — Proyecto: modelo trazable de extremo a extremo

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** Total = 100. accuracy = 75/100 = **0.75**; precisión = 30/45 =
**0.667**; recall = 30/40 = **0.75**; F1 = 2·(0.667·0.75)/(0.667+0.75) = **0.706**.
Baseline mayoritario (clase negativa, 60 casos): accuracy 0.60 — el modelo aporta
15 puntos, pero un tercio de sus alarmas son falsas (precisión 0.667): qué métrica
manda depende del costo de cada error.

**Ejercicio 2.** Las copias rotadas de una misma imagen quedaron repartidas entre
train y test: el modelo "reconoce" en test variantes casi idénticas de lo que
memorizó en train. La partición correcta se hace sobre las imágenes *originales*
(o mejor, sobre el grupo que las genera: paciente, usuario, sesión) **antes** de
cualquier aumento, que se aplica solo dentro de train.

**Ejercicio 3.** Una card honesta con este laboratorio: propósito = demo educativa de
pipeline trazable; datos = sintéticos generados con semilla 60; métrica = la que
declare el JSON (con su valor exacto); limitación = datos sintéticos y escala mínima;
uso desaconsejado = cualquier decisión real. Lo esencial: ningún campo afirma más de
lo que `evidence` contiene, y `limitations` pasa a la card sin suavizarse.

**Ejercicio 4.** Media = **0.70**; desviación poblacional = √(Σ(x−0.70)²/5) =
√0.00148/… = **0.0271**. Reportar "0.74" es cherry-picking de la mejor semilla; lo
honesto es **0.70 ± 0.03 (5 semillas)** — y dejar las cinco corridas registradas.


In [ ]:
result = run_lab("capstone", seed=60)
assert result["kind"] == "capstone"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
import math

# Ejercicio 1
TP, FP, FN, TN = 30, 15, 10, 45
total = TP + FP + FN + TN
accuracy = (TP + TN) / total
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)
print(f"acc={accuracy:.3f}  prec={precision:.3f}  rec={recall:.3f}  F1={f1:.3f}")
assert abs(f1 - 0.706) < 1e-3

# Ejercicio 4
accs = [0.71, 0.69, 0.74, 0.70, 0.66]
media = sum(accs) / len(accs)
desv = math.sqrt(sum((a - media) ** 2 for a in accs) / len(accs))
print(f"accuracy = {media:.2f} ± {desv:.4f} (n={len(accs)})")
assert abs(media - 0.70) < 1e-9


## Reflexión

1. De la cadena datos→splits→config→checkpoint→métrica, ¿qué eslabón registra este laboratorio y cuáles tendrías que añadir tú para un proyecto real?
2. ¿Por qué "elegir la mejor de 20 corridas" produce una estimación sesgada, y cómo la corrige el protocolo media ± σ?
3. Escribe dos afirmaciones sobre un modelo tuyo: una que tu evidencia actual soporta y otra que requeriría experimentos adicionales. ¿Qué las distingue?
